Get acticles from the actual google new categories to be used as a ground truth for trianing a classifier model

In [ ]:
import feedparser
import pandas as pd
import os
from datetime import datetime

# currently (06.06.26) valid topic ids for rss google news query
#ids may change at any point, failed requests are not cought here and will cause crashing or empty streams once goolge changes the internal topic id, which they will
TOPIC_IDS = {
    "general": "CAAqIQgKIhtDQkFTRGdvSUwyMHZNRE0wTldnU0FtUmxLQUFQAQ",      # Tab: Deutschland
    "business": "CAAqJggKIiBDQkFTRWdvSUwyMHZNRGx6TVdZU0FtUmxHZ0pFUlNnQVAB", # Tab: Unternehmen und Märkte
    "science": "CAAqKAgKIiJDQkFTRXdvSkwyMHZNR1ptZHpWbUVnSmtaUm9DUkVVb0FBUAE",  # Tab: Wissenschaft und Technik
    "health": "CAAqIQgKIhtDQkFTRGdvSUwyMHZNR3QwTlRFU0FtUmxLQUFQAQ",          # Tab: Gesundheit
}

FILENAME = "google_news_training_ground_truth.csv"

def clean_title_and_source(raw_title):
    if not raw_title:
        return "No Title", "unknown"
    if ' - ' in raw_title:
        parts = raw_title.rsplit(' - ', 1)
        return parts[0].strip(), parts[1].strip()
    return raw_title.strip(), "unknown"

def run_data_search():
    new_samples = []
    
    print("Collecting current articles from rss feed...")
    for cat_name, topic_id in TOPIC_IDS.items():
        rss_url = f"https://news.google.com/rss/topics/{topic_id}?hl=de&gl=DE&ceid=DE:de"
        feed = feedparser.parse(rss_url)
        print(f"-> Category '{cat_name}': {len(feed.entries)} articles in currend feed")
        
        for entry in feed.entries:
            titel, source = clean_title_and_source(entry.title)
            
            new_samples.append({
                "text": titel,
                "source": source,
                "label": cat_name,
                "original_url": entry.link,
                "date_raw": entry.published,
                "scraped_at": datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            })
            
    df_new = pd.DataFrame(new_samples)
        
    #check for existing train data file 
    if os.path.exists(FILENAME):
        print(f"\nFound existing train data '{FILENAME}', using file")
        df_old = pd.read_csv(FILENAME)
        print("\nDistribution of classes in found file:")
        print(df_old['label'].value_counts())
        
        #append newly found data to data from existing file  
        df_total = pd.concat([df_old, df_new], ignore_index=True)
        print(f"\nEntries before dupe pruning: {len(df_total)}")
    else:
        print(f"\nNo existing train data file found. Creating a new one in current directory: '{FILENAME}'")
        df_total = df_new

    #remove duplicates
    #im using the field "text" as the desciding criterion as the urls from google news can be different even though the point to the same article in some cases
    #'keep="first"' to ensure the oldest entrie with the same title is kept t
    entries_before_pruning = len(df_total)
    df_total = df_total.drop_duplicates(subset=['text'], keep='first')
    entries_after_pruning = len(df_total)
    
    df_total.to_csv(FILENAME, index=False, encoding="utf-8")

    #misc. eval output
    print("\n=== PROCESS COMPLETE ===")
    print(f"Newly added unique entries this run: {entries_after_pruning - (entries_before_pruning - len(df_new))}")
    print(f"New total number of entries: {entries_after_pruning}")
    print(f"Found and blocked duplicates: {entries_before_pruning - entries_after_pruning}")
    
    print("\nNew distribution of classes:")
    print(df_total['label'].value_counts())


In [5]:
run_data_search()

-> Category 'general': 12 articles in currend feed
-> Category 'business': 70 articles in currend feed
-> Category 'science': 38 articles in currend feed
-> Category 'health': 70 articles in currend feed

Found existing train data 'google_news_training_ground_truth.csv', using file

Distribution of classes in found file:
business    2344
general     2216
health      1450
science      786
Name: label, dtype: int64

Entries before dupe pruning: 6986

=== PROCESS COMPLETE ===
Newly added unique entries this run: 187
New total number of entries: 6983
Found and blocked duplicates: 3

New distribution of classes:
business    2413
general     2228
health      1519
science      823
Name: label, dtype: int64
